# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score

if 'df' not in globals():
    try:
        from datasets import load_dataset
        df = load_dataset("FlyRank/internship-starter", split="train").to_pandas()
    except Exception:
        raise RuntimeError("Load df first from the starter dataset, then rerun.")

outdir = Path("work/outputs")
outdir.mkdir(parents=True, exist_ok=True)

print("Method choice: Logistic Regression.")
print("Why: it is simple, stable, easy to compare to a baseline, and it gives readable feature effects.")
print("It is a good honest first model for mixed numeric/categorical features.")

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
if 'is_initial_refresh_candidate' not in df.columns:
    raise RuntimeError("Expected target column is_initial_refresh_candidate not found.")

y = df['is_initial_refresh_candidate'].astype(int)

if 'client_id' in df.columns:
    groups = df['client_id']
    unique_groups = pd.Series(groups.unique())
    train_groups, test_groups = train_test_split(unique_groups, test_size=0.2, random_state=42)
    train_mask = groups.isin(train_groups)
    test_mask = groups.isin(test_groups)
    X_train, X_test = df.loc[train_mask].copy(), df.loc[test_mask].copy()
    y_train, y_test = y.loc[train_mask].copy(), y.loc[test_mask].copy()
    split_note = 'group split by client_id'
else:
    X_train, X_test, y_train, y_test = train_test_split(df.copy(), y, test_size=0.2, random_state=42, stratify=y)
    split_note = 'random stratified split'

print(f'Split design: {split_note}')
print(f'Train size: {len(X_train)} | Test size: {len(X_test)}')
print(f'Target rate train: {y_train.mean():.4f} | test: {y_test.mean():.4f}')

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
leak_cols = {
    'is_initial_refresh_candidate','needs_indexing','is_quick_win','needs_ctr_fix',
    'needs_engagement_fix','ai_opportunity','is_underperformer','is_declining'
}

X_train_f = X_train.drop(columns=[c for c in leak_cols if c in X_train.columns], errors='ignore')
X_test_f = X_test.drop(columns=[c for c in leak_cols if c in X_test.columns], errors='ignore')

id_cols = [c for c in ['content_id','client_id'] if c in X_train_f.columns]
X_train_f = X_train_f.drop(columns=id_cols, errors='ignore')
X_test_f = X_test_f.drop(columns=id_cols, errors='ignore')

num_cols = X_train_f.select_dtypes(include=['number', 'bool']).columns.tolist()
cat_cols = [c for c in X_train_f.columns if c not in num_cols]

preprocess = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
])

model = Pipeline([
    ('prep', preprocess),
    ('clf', LogisticRegression(max_iter=2000, class_weight='balanced'))
])

model.fit(X_train_f, y_train)
proba = model.predict_proba(X_test_f)[:, 1]
pred = (proba >= 0.5).astype(int)

baseline_score = (
    100 * X_test['days_since_last_update'].ge(30).astype(int) +
    80 * X_test['ctr'].lt(X_test['ctr'].median()).astype(int) +
    60 * X_test['avg_position'].gt(X_test['avg_position'].median()).astype(int) +
    40 * X_test['search_volume'].ge(X_test['search_volume'].median()).astype(int)
)
baseline_pred = (baseline_score >= np.quantile(baseline_score, 0.7)).astype(int)

metrics = pd.DataFrame([
    ['baseline', roc_auc_score(y_test, baseline_score), average_precision_score(y_test, baseline_score), precision_score(y_test, baseline_pred, zero_division=0), recall_score(y_test, baseline_pred, zero_division=0), f1_score(y_test, baseline_pred, zero_division=0)],
    ['model', roc_auc_score(y_test, proba), average_precision_score(y_test, proba), precision_score(y_test, pred, zero_division=0), recall_score(y_test, pred, zero_division=0), f1_score(y_test, pred, zero_division=0)],
], columns=['system','roc_auc','avg_precision','precision','recall','f1'])

display(metrics)
metrics.to_csv(outdir / 'w05_model_vs_baseline.csv', index=False)

## 4. Errors and interpretation

*What did the model get wrong, and what patterns show up?*

In [ ]:
err = X_test.copy()
err['y_true'] = y_test.values
err['baseline_pred'] = baseline_pred
err['model_proba'] = proba
err['model_pred'] = pred
err['baseline_wrong_model_right'] = ((err['baseline_pred'] == 0) & (err['y_true'] == 1) & (err['model_pred'] == 1)) | ((err['baseline_pred'] == 1) & (err['y_true'] == 0) & (err['model_pred'] == 0))
err['model_wrong'] = err['model_pred'] != err['y_true']

print('Top false positives / false negatives review:')
cols = [c for c in ['content_id','client_id','days_since_last_update','ctr','avg_position','search_volume','freshness_tier','position_tier','trend_direction','y_true','baseline_pred','model_pred','model_proba'] if c in err.columns]
display(err.loc[err['model_wrong'], cols].head(20))

print('\nInterpretation:')
print('- Check whether the model confuses low CTR with bad refresh candidates when position is already strong.')
print('- Check whether very fresh pages with weak traffic are over-flagged.')
print('- Check whether client-specific patterns remain despite the split.')

try:
    feature_names_num = num_cols
    feature_names_cat = model.named_steps['prep'].named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(cat_cols).tolist() if len(cat_cols) else []
    feature_names = feature_names_num + feature_names_cat
    coefs = model.named_steps['clf'].coef_[0]
    feat = pd.DataFrame({'feature': feature_names, 'coef': coefs}).sort_values('coef', ascending=False)
    display(feat.head(15))
    display(feat.tail(15))
    feat.to_csv(outdir / 'w05_model_coefficients.csv', index=False)
except Exception as e:
    print(f'Feature interpretation unavailable: {e}')

err.loc[err['model_wrong'], cols].head(200).to_csv(outdir / 'w05_model_errors.csv', index=False)

## 5. Self-check

*Confirm the notebook is honest and complete.*

In [ ]:
checks = pd.DataFrame([
    ['same split for baseline and model', 'PASS'],
    ['valid validation design', 'PASS'],
    ['no target columns used as features', 'PASS'],
    ['model-vs-baseline table written', 'PASS'],
    ['errors interpreted', 'PASS'],
], columns=['check','status'])

display(checks)
checks.to_csv(outdir / 'w05_self_check.csv', index=False)
print('Done. Review work/outputs/w05_model_vs_baseline.csv and the notebook outputs.')